In [2]:
import cv2
import numpy as np
import threading
import time

class Camera:
    def __init__(self, width=640, height=480):
        self.width = width
        self.height = height
        self._running = False
        self._thread = None
        self.frame = np.zeros((height, width, 3), dtype=np.uint8)

        # --- ЭТОТ КОНВЕЙЕР ДОЛЖЕН ЗАРАБОТАТЬ ---
        # Он берет видео с камеры и преобразует его в формат BGR, который нужен OpenCV
        gst_pipeline = (
            f"nvarguscamerasrc ! "
            f"video/x-raw(memory:NVMM),width={width},height={height},framerate=30/1 ! "
            f"nvvidconv ! video/x-raw,format=BGRx ! "
            f"videoconvert ! video/x-raw,format=BGR ! "
            f"appsink drop=1"
        )
        # -----------------------------------------

        print(f"Попытка открыть камеру с конвейером:\n{gst_pipeline}")
        self.cap = cv2.VideoCapture(gst_pipeline, cv2.CAP_GSTREAMER)

        if not self.cap.isOpened():
            print("❌ КРИТИЧЕСКАЯ ОШИБКА: Не удалось открыть камеру через GStreamer.")
            print("Это значит, что OpenCV не может использовать GStreamer pipeline.")
            print("Проверьте, что OpenCV собран с поддержкой GStreamer, или используйте другой метод.")
            # Здесь можно создать заглушку или просто оставить черный кадр
        else:
            print("✅ Камера успешно открыта через GStreamer.")

    def start(self):
        self._running = True
        self._thread = threading.Thread(target=self._update)
        self._thread.daemon = True
        self._thread.start()
        time.sleep(1.0)
        print("📷 Поток захвата кадров запущен")

    def _update(self):
        while self._running:
            ret, frame = self.cap.read()
            if ret and frame is not None:
                self.frame = frame
            else:
                time.sleep(0.01)

    def read(self):
        return self.frame

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(0.5)
        if hasattr(self, 'cap') and self.cap:
            self.cap.release()
        print("🛑 Камера остановлена")